# LDPC码通用执行过程：编码与迭代译码的标准步骤

在不失一般性的前提下，以下是教材中关于一个参数为 $(N, K)$、校验矩阵为 $M \times N$ 的 LDPC 码的完整系统执行流程序列。

## 阶段一：系统初始化
1. **构造校验矩阵**：根据设计准则（如度分布多项式或有限几何）生成稀疏的 $M \times N$ 奇偶校验矩阵 $H$。
2. **推导生成矩阵**：为消除代数运算的歧义，通过引入显式的变换矩阵将原矩阵转化为系统码形式。
   * **满秩假设**：假设 $H$ 为满行秩矩阵，即 $\text{rank}(H) = M$，此时 $K = N - M$。（注：若生成的 $H$ 存在线性相关行，即冗余校验方程，需预先剔除，保证其满秩后再进行后续推导）。
   * **列置换与基底选择**：由于 $\text{rank}(H) = M$，必然可以在 $H$ 中找到 $M$ 个线性无关的列。定义 $N \times N$ 的正交置换矩阵 $\Pi$ 将这 $M$ 个线性无关的列置换到矩阵的最右侧。此时有 $H\Pi = [A_{M \times K} \mid B_{M \times M}]$，其中右侧的方阵 $B$ 由线性无关列组成，必定满秩且可逆。
   * **行变换化零**：定义 $M \times M$ 的可逆满秩矩阵 $T = B^{-1}$ 表示所有初等行变换的累积。
   * 将原校验矩阵 $H$ 转换为**系统校验矩阵** $H_{sys}$（与原矩阵 $H$ 严格区分）：$H_{sys} = T H \Pi = B^{-1} [A \mid B] = [B^{-1}A \mid I_{M \times M}] = [P_{M \times K} \mid I_{M \times M}]$ *(其中 $I_{M \times M}$ 为 $M$ 阶单位阵，$P_{M \times K} = B^{-1}A$ 为非系统部分的稠密矩阵)*
   * 构造对应的**系统生成矩阵** $G_{sys}$：$G_{sys} = [I_{K \times K} \mid P^T_{K \times M}]$。*(正交性证明：在 $\text{GF}(2)$ 域中，执行分块矩阵乘法可得 $G_{sys}H_{sys}^T = [I_{K \times K} \mid P^T] [P^T \mid I_{M \times M}]^T = I_{K \times K}P^T + P^T I_{M \times M} = P^T + P^T = \mathbf{0} \pmod 2$)*
   * 恢复原始变量顺序得到最终的**生成矩阵** $G$：由于列置换矩阵满足 $\Pi^{-1} = \Pi^T$，定义 $G = G_{sys} \Pi^T$。该 $G$ 严格满足 $GH^T = \mathbf{0}$。

## 阶段二：编码执行过程 (Encoding)
**输入**：长度为 $K$ 的原始信息比特向量 $\mathbf{u} = [u_1, u_2, \dots, u_K]$。
**执行步骤**：
1. 执行矩阵乘法运算：$\mathbf{c} = \mathbf{u}G \pmod 2$。
2. 得到长度为 $N$ 的码字向量 $\mathbf{c} = [c_1, c_2, \dots, c_N]$。由于可能经历了列置换 $\Pi$，码字的系统位可能不再严格位于前 $K$ 位，而是分散在 $N$ 位中。

## 阶段二（扩展）：稀疏校验矩阵的工程困境与主流高效编码方案

如果按照上述“阶段一”的逻辑，完全随机地构造一个满秩的稀疏校验矩阵 $H$，并在“阶段二”使用 $\mathbf{c} = \mathbf{u}G$ 进行直接编码，会面临极其严重的工程瓶颈：**稠密生成矩阵问题**。

尽管 $H$ 是极度稀疏的，但通过高斯消元法求逆（$T=B^{-1}$）得到的非系统部分矩阵 $P = B^{-1}A$ 往往是**极其稠密**的。这意味着生成的 $G_{sys}$ 也是稠密的。此时，计算 $\mathbf{c} = \mathbf{u}G$ 的编码复杂度为 $\mathcal{O}(K \times (N-K))$，即 $\mathcal{O}(N^2)$。对于动辄几万甚至几十万码长的现代通信系统，这种平方级的编码复杂度在芯片上是不可接受的。因此，现代标准不再使用随机稀疏矩阵，而是人为设计具有特殊结构的 $H$ 以实现线性时间 $\mathcal{O}(N)$ 编码。

### 方案一：Richardson-Urbanke (RU) 算法 (近似线性时间编码)
**H矩阵结构设计**：不再强制化为严格的系统形式 $[P \mid I]$，而是通过纯粹的行列置换矩阵 $P_{row}$ 和 $P_{col}$，将原稀疏矩阵 $H$ 转化为近似下三角形式（Approximate Lower Triangular, ALT）的新矩阵 $H_{ALT}$：
$H_{ALT} = P_{row} H P_{col} = \begin{bmatrix} A_{(M-g) \times K} & B_{(M-g) \times g} & T_{(M-g) \times (M-g)} \\ C_{g \times K} & D_{g \times g} & E_{g \times (M-g)} \end{bmatrix}$
*(注：设码字长度为 $N$，信息位长度为 $K$，校验位长度为 $M = N - K$。$g$ 为间隙部分大小。其中 $T$ 为极度稀疏的下三角方阵。)*

**如何化为下三角阵 $T$ (严格的 RU 贪心三角化算法)**：
为了保持矩阵极度稀疏，该步骤**仅允许纯粹的行列置换**。由于 $T$ 是**下三角矩阵**（主对角线右上方全为 0），我们必须通过不断寻找“残余行重为 1 的行”来从上到下构建它。严格执行步骤如下：
1. **寻找行重为 1 的行 (Find Residual Row Weight 1)**：在当前尚未处理的矩阵区域（初始为整个 $H$ 矩阵）中，扫描寻找只包含一个“1”的行。
2. **对角线安置 (Diagonal Placement)**：将找到的这行交换到当前处理区域的第一行，并将那个唯一的“1”所在的列交换到第一列。由于该行在当前区域内只有一个 1，所以该“1”右侧的元素必定全为 0。这严格满足了下三角矩阵第一行的结构要求（此时这个 1 成为 $T$ 的对角线元素）。
3. **降阶与连锁反应 (Reduction & Avalanche)**：将这第一行和第一列从“未处理区域”中剥离（它们已正式确认属于 $T$ 矩阵的成分）。**核心机制**：由于我们从系统中划去了一整列，剩余未处理矩阵中与该列相交的行的 1 的个数会随之减少（即残余行重降低）。这种降低会触发“连锁反应”，暴露出新的行重为 1 的行。回到步骤 1 循环执行。
4. **卡壳与间隙生成 (Stagnation & Gap Creation)**：如果某一步发现当前未处理区域没有任何行重为 1 的行了（即剩余所有行的残余行重均 $\ge 2$），贪心过程“卡壳”。此时必须妥协：在未处理的列中主动挑出一列（通常启发式地选择列重最大的列），**将其彻底踢出后续的下三角构造过程，划归到间隙（Gap）区域**（即这列最终会变成左侧矩阵 $B$ 和 $D$ 的一部分）。剔除这列后，剩余相关行的残余行重会强制减小，从而极大概率重新产生行重为 1 的行，解开卡壳状态。每一次这样的“妥协踢出”操作，都会使得最终的间隙大小 $g$ 增加 1。反复执行直到划分完成。

**关于间隙大小 $g$ 的理论前提与上界**：
需要特别澄清：RU 论文**并没有**证明“任何稀疏满秩矩阵都能做到 $g = \mathcal{O}(\sqrt{N})$”。实际上，如果人为构造一个拓扑结构极其恶劣（例如内部高度连通、无法通过降阶触发连锁反应）的稀疏矩阵，$g$ 完全可能退化到与 $N$ 同阶的 $\mathcal{O}(N)$。
RU 定理是一个**基于概率系综（Ensemble）的渐近结果**。其严格表述为：对于由一对特定的、经过优化的度分布多项式 $(\lambda, \rho)$ 随机生成的 LDPC 码族，当满足一定设计条件时，随着码长 $N \to \infty$，贪心算法陷入卡壳的概率极低，平均间隙大小 $g$ 的期望值为 $\mathcal{O}(\sqrt{N})$。换言之，是**“精心设计的度分布特性”加上“稀疏性”**共同保证了 $g \ll N$ 的结果。在工程实践中（例如 $N=10^4 \sim 10^5$），对于优化好的随机 LDPC 码，$g$ 通常只有数十的量级。此时唯一涉及稠密矩阵乘法的复杂度 $g^2 \approx \mathcal{O}(N)$，完美保证了总编码复杂度为近似线性时间 $\mathcal{O}(N)$。

**RU编码执行流程与公式推导**：
将对应于新矩阵 $H_{ALT}$ 的置换后码字向量 $\mathbf{c}_{ALT}$ 划分为三部分：$\mathbf{c}_{ALT} = [\mathbf{u}, \mathbf{p}_1, \mathbf{p}_2]$，其中 $\mathbf{u}$ 为系统信息位（长度 $K$），$\mathbf{p}_1$ 为第一部分校验位（长度 $g$），$\mathbf{p}_2$ 为第二部分校验位（长度 $M-g$）。

**有效性证明（为什么这样定义满足编码需求）**：
为了使 $\mathbf{c}_{ALT}$ 成为合法的码字，必须满足核心校验方程 $H_{ALT} \mathbf{c}_{ALT}^T = \mathbf{0}$。展开该分块矩阵乘法：
$\begin{bmatrix} A & B & T \\ C & D & E \end{bmatrix} \begin{bmatrix} \mathbf{u}^T \\ \mathbf{p}_1^T \\ \mathbf{p}_2^T \end{bmatrix} = \begin{bmatrix} \mathbf{0} \\ \mathbf{0} \end{bmatrix}$
这等价于求解以下两个联立的矩阵方程组：
(1) $A\mathbf{u}^T + B\mathbf{p}_1^T + T\mathbf{p}_2^T = \mathbf{0}$
(2) $C\mathbf{u}^T + D\mathbf{p}_1^T + E\mathbf{p}_2^T = \mathbf{0}$

由方程 (1)，由于 $T$ 是下三角满秩方阵，可直接用前向代入法解出 $\mathbf{p}_2^T$：
$T\mathbf{p}_2^T = -A\mathbf{u}^T - B\mathbf{p}_1^T \implies \mathbf{p}_2^T = -T^{-1}(A\mathbf{u}^T + B\mathbf{p}_1^T)$

将 $\mathbf{p}_2^T$ 的表达式代入方程 (2) 中以消去 $\mathbf{p}_2$：
$C\mathbf{u}^T + D\mathbf{p}_1^T - E T^{-1}(A\mathbf{u}^T + B\mathbf{p}_1^T) = \mathbf{0}$

合并 $\mathbf{u}^T$ 与 $\mathbf{p}_1^T$ 的同类项：
$(-E T^{-1}A + C)\mathbf{u}^T + (-E T^{-1}B + D)\mathbf{p}_1^T = \mathbf{0}$

此时，定义核心稠密矩阵 $\Phi = -E T^{-1}B + D$，上式即化简为：
$\Phi \mathbf{p}_1^T = -(-E T^{-1}A + C)\mathbf{u}^T \implies \mathbf{p}_1^T = -\Phi^{-1} (-E T^{-1}A + C)\mathbf{u}^T$
这就完美推导出了“仅需对规模极小的矩阵 $\Phi$ 求逆即可求解 $\mathbf{p}_1$”的结论。

基于上述严密推导，真正的在线编码步骤如下（注：最终发送时需通过 $P_{col}^T$ 恢复原始变量顺序。在 $\text{GF}(2)$ 中，所有的减号 $-$ 等价于加号 $+$）：
1. **离线预计算 (Offline Precomputation)**：由于 $H$ 矩阵结构一旦确定就不再改变，常数矩阵 $\Phi = E T^{-1} B + D$ 及其逆矩阵 $\Phi^{-1}$ 可以在设备出厂前一次性预先计算并固化在存储中。这属于一次性的离线开销，不计入实时的在线编码过程。
2. **在线编码 - 计算中间变量（降复杂度的核心机密）**：如果将 $(E T^{-1}A + C)$ 作为整体预计算，它将是一个巨大的稠密矩阵，导致计算退化为 $\mathcal{O}(N^2)$。因此必须将其拆解，利用 $A, C, E, T$ 的稀疏性质，依次计算一系列中间向量：
   * **步骤 2.a**: 计算向量 $\mathbf{a}^T = A\mathbf{u}^T$。*(极度稀疏的矩阵 $A$ 乘以向量，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.b**: 计算向量 $\mathbf{b}^T = T^{-1}\mathbf{a}^T$。*(注意：并非真正对 $T$ 求逆，而是利用 $T$ 的下三角特性求解方程 $T\mathbf{b}^T = \mathbf{a}^T$，通过极其高效的**前向代入（Forward Substitution）**得出，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.c**: 计算向量 $\mathbf{c}^T = C\mathbf{u}^T$。*(稀疏乘法，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.d**: 计算向量 $\mathbf{d}^T = E\mathbf{b}^T$。*(稀疏乘法，复杂度 $\mathcal{O}(N)$)*
   * **步骤 2.e**: 得到总中间向量 $\mathbf{e}^T = \mathbf{d}^T + \mathbf{c}^T$。*(即公式推导中的 $(-E T^{-1}A + C)\mathbf{u}^T$ 部分。所有前置运算均为 $\mathcal{O}(N)$)*
3. **在线编码 - 求解 $\mathbf{p}_1$**：根据推导公式计算 $\mathbf{p}_1^T = \Phi^{-1} \mathbf{e}^T$。由于 $\Phi^{-1}$ 是维度仅为 $g \times g$ 的常数矩阵，此处操作的在线计算复杂度严格控制在 $\mathcal{O}(g^2)$ 级别。
4. **在线编码 - 求解 $\mathbf{p}_2$**：再次利用 $T$ 的下三角特性，根据方程(1)计算向量 $\mathbf{f}^T = A\mathbf{u}^T + B\mathbf{p}_1^T = \mathbf{a}^T + B\mathbf{p}_1^T$。随后求解方程 $T \mathbf{p}_2^T = \mathbf{f}^T$，通过前向代入解出 $\mathbf{p}_2$。至此，整体在线编码复杂度严格降至 $\mathcal{O}(N + g^2)$。

### 方案二：准循环 LDPC (QC-LDPC) 双对角结构 (目前最主流，如 5G NR, Wi-Fi 802.11n)
**H矩阵结构设计**：这是目前工业界的绝对主流。$H$ 矩阵不再按单个比特随机生成，而是由大小为 $Z \times Z$ 的循环置换矩阵（Circulant Permutation Matrices, CPM）或全零矩阵块拼接而成（Base Graph 展开）。更关键的是，其校验部分 $H_p$ 在设计之初就被严格构造为**双对角结构（Dual-Diagonal）**或阶梯状下三角结构。因此原矩阵直接呈现为 $H = [H_u \mid H_p]$ 的天然分块形式。
**QC-LDPC编码执行流程**：
直接利用设计好的 $H = [H_u \mid H_p]$，码字 $\mathbf{c} = [\mathbf{u}, \mathbf{p}]$。要求满足 $H\mathbf{c}^T = H_u \mathbf{u}^T + H_p \mathbf{p}^T = \mathbf{0} \pmod 2$，即 $H_p \mathbf{p}^T = H_u \mathbf{u}^T$。
1. 首先计算系统信息引发的校验和向量 $\mathbf{v} = H_u \mathbf{u}^T$。由于 $H_u$ 同样是稀疏且由循环块构成的，这一步可通过简单的移位和稀疏异或完成，复杂度 $\mathcal{O}(N)$。
2. 求解双对角方程：由于 $H_p$ 呈现双对角线全为 $1$（类似 $p_1 + p_2 = v_1$，$p_2 + p_3 = v_2 \dots$ 的差分方程结构），求解向量 $\mathbf{p}$ 变成了一个简单的累加器（Accumulator）过程：
   * $p_1$ 可通过结构中的某个环（如 5G 中的 Core Graph 特定行）单独解出。
   * 后续校验比特直接递推：$p_i = p_{i-1} + v_{i-1} \pmod 2$。
**结论**：在 QC-LDPC 架构下，**完全不需要**计算或存储稠密的生成矩阵 $G$。编码器仅需要移位寄存器和异或门，通过依次累加即可完成严格 $\mathcal{O}(N)$ 复杂度的编码工作。

### 深入解析：QC-LDPC (准循环 LDPC) 的严格代数结构

在现代通信标准（如 5G NR, Wi-Fi 802.11n/ac/ax, DVB-S2）中，QC-LDPC 因为其高度适配硬件并行处理的特性而成为绝对主流。它的校验矩阵 $H$ 具有严格的两级分层代数结构：**宏观基图（Base Graph）**与**微观循环阵（Circulant Permutation Matrices, CPM）**。

#### 1. 两级映射与提升机制 (Lifting)
QC-LDPC 的矩阵并非按单个比特随机生成，而是通过将一个小规模的**基础矩阵**（Base Matrix，或称基图）按照**提升因子**（Lifting Factor, 记为 $Z$）进行“膨胀”（Lifting）得到的。

设基图矩阵为 $H_B$，维度为 $m_b \times n_b$。提升因子为 $Z$。
最终生成的实际二元校验矩阵 $H$ 的维度为 $M \times N$，其中 $M = m_b \times Z$，$N = n_b \times Z$。

基图 $H_B$ 中的每一个元素（设为 $P_{i,j}$）都代表了一个规模为 $Z \times Z$ 的微观子矩阵：
*   **空缺（Null / -1）**：如果 $P_{i,j} = -1$（部分文献用 $\infty$ 表示），则它在实际 $H$ 矩阵中被展开为一个 $Z \times Z$ 的**全零矩阵** $\mathbf{0}_{Z \times Z}$。
*   **循环移位（Shift Value）**：如果 $P_{i,j} \ge 0$，则它被展开为一个**标准单位阵经过循环右移** $P_{i,j}$ 位后得到的 $Z \times Z$ 循环置换矩阵 $I(P_{i,j})$。
    *   例如，$P_{i,j} = 0$ 代表未移位的标准单位阵 $I_{Z \times Z}$（主对角线为 1）。
    *   例如，$P_{i,j} = 1$ 代表第一行移到第二行、最后一行移到第一行的循环右移矩阵。

由此，实际的校验矩阵 $H$ 呈现出宏观上的分块循环结构：
$H = \begin{bmatrix} I(P_{0,0}) & I(P_{0,1}) & \cdots & I(P_{0, n_b-1}) \\ I(P_{1,0}) & I(P_{1,1}) & \cdots & I(P_{1, n_b-1}) \\ \vdots & \vdots & \ddots & \vdots \\ I(P_{m_b-1, 0}) & I(P_{m_b-1, 1}) & \cdots & I(P_{m_b-1, n_b-1}) \end{bmatrix}$

#### 2. 宏观结构：系统信息与校验的天然分离
在设计基图 $H_B$ 时，标准会人为将其从列的方向上严格切分为两部分：$H_B = [H_{B,u} \mid H_{B,p}]$。
*   $H_{B,u}$ 对应**系统信息部分（Systematic Part）**，其移位值设计用于优化变量节点度分布以逼近香农极限（通常这部分列重较高，包含打孔位以提供高增益）。
*   $H_{B,p}$ 对应**校验部分（Parity Part）**，这部分被严格限制为极其特殊的结构（通常是**双对角**或**阶梯下三角**），从而**彻底消灭**了传统 LDPC 编码时的矩阵求逆与稠密乘法过程。

#### 3. 校验部分 ($H_{B,p}$) 的双对角（Dual-Diagonal）核心结构
以 IEEE 802.11n 和 5G NR 的基础结构为例，$H_{B,p}$ 最核心的 Core Parity 基图结构呈现如下形式：

$H_{B,p\_core} = \begin{bmatrix} P_{0,0} & 0 & -1 & \cdots & -1 \\ P_{1,0} & 0 & 0 & \cdots & -1 \\ -1 & -1 & 0 & \ddots & -1 \\ \vdots & \vdots & \ddots & \ddots & 0 \\ -1 & -1 & \cdots & -1 & 0 \end{bmatrix}$

在这个结构中：
1.  **阶梯双对角线**：除了第一列特殊外，主对角线及其下方的次对角线上的元素全为 $0$（代表未移位的 $Z \times Z$ 单位阵 $I$）。这在展开后形成了两条平行的单位阵对角线。
2.  **全零区域**：右上角和左下角的元素全为 $-1$（代表全零矩阵）。
3.  **闭环破缺列（第一列）**：第一列包含特定的移位值（如 $P_{0,0}, P_{1,0}$），这是为了消除双对角矩阵直接相加可能导致的秩亏损（Rank Deficiency），确保整个校验矩阵满秩。

#### 4. 为什么 QC-LDPC 能够实现严格的 $\mathcal{O}(N)$ 硬件极速编码？
当我们要满足核心校验方程 $H\mathbf{c}^T = \mathbf{0}$，即 $[H_u \mid H_p] [\mathbf{u}, \mathbf{p}]^T = \mathbf{0} \implies H_p \mathbf{p}^T = H_u \mathbf{u}^T$ 时：

1.  **计算伴随向量**：首先计算右侧的校验和向量 $\mathbf{v}^T = H_u \mathbf{u}^T$。由于 $\mathbf{u}$ 是已知信息位，且 $H_u$ 是由循环移位矩阵构成的。在硬件中，矩阵乘法 $I(P_{i,j}) \mathbf{u}^T$ 等价于**对向量 $\mathbf{u}$ 进行 $P_{i,j}$ 位的循环移位**。这完全可以通过极低功耗的移位寄存器网络和稀疏异或树在 $\mathcal{O}(N)$ 内并行计算完毕。
2.  **累加器极速求解 $\mathbf{p}$**：将求得的 $\mathbf{v}$ 代入方程 $H_p \mathbf{p}^T = \mathbf{v}^T$。我们将校验比特 $\mathbf{p}$ 按大小为 $Z$ 的块划分为 $\mathbf{p}_1, \mathbf{p}_2, \dots, \mathbf{p}_{m_b}$。由于 $H_p$ 的绝大部分是双单位阵构成的双对角结构，方程在微观上直接退化为极简的差分方程组：
    *   通过特殊的首列和末行预先消元解出第一块校验向量 $\mathbf{p}_1$。
    *   从第二块开始，后续微观方程形式全部呈现为：$\mathbf{p}_{i-1} + \mathbf{p}_{i} = \mathbf{v}_{i-1} \pmod 2$。
    *   此时，求解剩余所有校验块的过程变成了极其简单的前向**递推异或累加（Accumulation）**：$\mathbf{p}_{i} = \mathbf{p}_{i-1} + \mathbf{v}_{i-1} \pmod 2$。

**理论结论**：QC-LDPC 通过“基图设定宏观双对角结构 + 提升因子决定微观循环移位矩阵”的精妙代数设计，既保证了宏观上无需预先计算或存储任何稠密矩阵 $\Phi$ 或 $G$（实现纯粹的 $\mathcal{O}(N)$ 复杂度），又保证了微观上极度规整、避免了存储器读写冲突（Memory Conflict），完美解决了现代通信芯片在 Tbps 级吞吐量下的编码和译码并行化刚需。

## 阶段三：信道映射与传输 (Transmission)
**输入**：码字 $\mathbf{c}$。
**执行步骤**：
1. **调制映射**：将二进制码字映射为物理信道符号（以BPSK为例），执行变换 $x_i = 1 - 2c_i$。此时 $0 \to +1$，$1 \to -1$。
2. **信道衰减与加噪**：信号通过信道，接收端收到实数向量 $\mathbf{y} = [y_1, y_2, \dots, y_N]$。在AWGN信道下，$y_i = x_i + n_i$，其中 $n_i \sim \mathcal{N}(0, \sigma^2)$。
3. **计算初始对数似然比 (LLR)**：基于接收到的连续值 $y_i$ 和信道噪声方差 $\sigma^2$，计算每个变量节点 $v_j$ 的先验LLR：$L^{(0)}(v_j) = \ln \frac{P(c_j=0 | y_j)}{P(c_j=1 | y_j)} = \frac{2y_j}{\sigma^2}$

## 阶段四：置信传播 (BP) 迭代译码过程
**输入**：初始 LLR 向量 $L^{(0)}$，原始奇偶校验矩阵 $H$。
**执行步骤**：

**步骤 4.1：初始化 (Initialization)**
对于所有的 $(i, j)$ 满足 $H_{i,j} = 1$，将变量节点 $v_j$ 传递给校验节点 $c_i$ 的初始消息 $L_{v_j \to c_i}^{(0)}$ 设置为信道初始 LLR：$L_{v_j \to c_i}^{(0)} = L^{(0)}(v_j)$。设置迭代计数器 $l = 1$，最大迭代次数为 $l_{\max}$。

**步骤 4.2：校验节点更新 (Check Node Update, CNU)**
对于每个校验节点 $c_i$，根据其相连的变量节点（除目标节点 $v_j$ 外）传入的消息，计算传回给 $v_j$ 的外部信息 $L_{c_i \to v_j}^{(l)}$：$L_{c_i \to v_j}^{(l)} = 2 \tanh^{-1} \left( \prod_{j' \in \mathcal{N}(c_i) \setminus \{j\}} \tanh\left(\frac{L_{v_{j'} \to c_i}^{(l-1)}}{2}\right) \right)$ *(注：硬件实现中常用 Min-Sum 算法将其简化为符号相乘与寻找最小值的操作。)*

**步骤 4.3：变量节点更新 (Variable Node Update, VNU)**
对于每个变量节点 $v_j$，整合来自信道的先验信息与所有相连校验节点（除目标节点 $c_i$ 外）传来的外部信息，计算下一轮传递给 $c_i$ 的消息：$L_{v_j \to c_i}^{(l)} = L^{(0)}(v_j) + \sum_{i' \in \mathcal{N}(v_j) \setminus \{i\}} L_{c_{i'} \to v_j}^{(l)}$

**步骤 4.4：后验概率计算与硬判决 (Hard Decision)**
综合变量节点 $v_j$ 收到的所有信息（包括信道信息和所有校验节点信息），计算总的后验 LLR：$L_{total}(v_j) = L^{(0)}(v_j) + \sum_{i \in \mathcal{N}(v_j)} L_{c_i \to v_j}^{(l)}$。执行硬判决得到估计码字 $\hat{\mathbf{c}}$：
* 若 $L_{total}(v_j) \ge 0$，则判决 $\hat{c}_j = 0$
* 若 $L_{total}(v_j) < 0$，则判决 $\hat{c}_j = 1$

**步骤 4.5：伴随式校验与停止准则 (Syndrome Check)**
计算伴随式向量 $\mathbf{S} = \hat{\mathbf{c}}H^T \pmod 2$。
* 若 $\mathbf{S} = \mathbf{0}$，说明所有校验方程均满足，译码成功，跳出迭代。
* 若 $\mathbf{S} \neq \mathbf{0}$：
  * 若 $l < l_{\max}$，令 $l = l + 1$，返回**步骤 4.2** 继续下一次迭代。
  * 若 $l = l_{\max}$，说明达到最大迭代次数仍未收敛，声明译码失败 (Decoding Failure)。

## 阶段五：信息恢复
**输入**：译码成功的估计码字 $\hat{\mathbf{c}}$。
**执行步骤**：
将估计码字 $\hat{\mathbf{c}}$ 通过逆向的列置换操作 $\hat{\mathbf{c}}_{sys} = \hat{\mathbf{c}} \Pi$ 恢复为标准系统码字顺序，随后直接提取前 $K$ 个比特，即为恢复的原始信息 $\hat{\mathbf{u}} = [\hat{c}_{sys,1}, \hat{c}_{sys,2}, \dots, \hat{c}_{sys,K}]$。

### 附注：深入理解步骤 4.2 (校验节点更新，CNU)

步骤 4.2 是整个 LDPC 置信传播（BP）译码算法中最核心，也是数学形式最复杂的环节。我们可以从物理直觉、数学原理和工程实现三个层面来拆解这一步：

#### 1. 物理直觉：外信息 (Extrinsic Information) 传递
你可以把“校验节点”想象成一个**“会议室”**，它的唯一规则是：**所有参与这个会议的变量节点，它们的值的异或结果必须为 0（即总和为偶数）**。

当校验节点 $c_i$ 要给变量节点 $v_j$ 发送消息 $L_{c_i \to v_j}$ 时，这条消息的含义是：“根据除你之外其他所有人的表态，为了满足偶数规则，**我猜你应该是个什么值，以及我对此有多确定**。”

这就是为什么公式底下的连乘符号是 $\prod_{j' \in \mathcal{N}(c_i) \setminus \{j\}}$ —— 也就是**严格排除掉 $v_j$ 自己**。这避免了“回音壁效应”（把 $v_j$ 告诉 $c_i$ 的信息再原封不动地还给 $v_j$），防止节点产生正反馈而盲目自信，从而保证算法能够收敛。

#### 2. 数学原理：对数似然比 (LLR) 的加法同构变换
传递的信息 $L$ 是**对数似然比 (LLR)**。在概率论中，两个独立比特进行异或（模 2 加法）操作时，直接在 LLR 域计算联合概率非常复杂。但通过 $\tanh$ 函数可以实现一个极其美妙的变换（称为和积算法，Sum-Product Algorithm, SPA）：
*   **$\tanh\left(\frac{L}{2}\right)$ 映射**：将 LLR 完美地映射到 $[-1, 1]$ 的概率空间内。
*   **乘法等价**：在映射到这个空间后，**多个独立比特异或后的分布，直接等价于它们各自映射值的连乘！** 这就是公式中 $\prod$ 的由来。
*   **$2 \tanh^{-1}(\cdot)$ 反变换**：把乘出来的结果，重新拉回到 LLR 域，打包发送给 $v_j$。

#### 3. 工程实现：最小和算法 (Min-Sum Algorithm, MSA)
上述 SPA 算法在理论上极其完美，但在真实的通信芯片（如 5G 基带或 Wi-Fi 芯片）中，计算 $\tanh$ 和 $\tanh^{-1}$ 这种非线性超越函数会极其消耗硬件资源。工程师们发现，一堆数字的乘积，其最终的幅度其实**主要由这堆数字里面绝对值最小的那一个决定**（最不确定的那个节点决定了整个校验链条的可靠性下限）。

因此，硬件实现中将步骤 4.2 简化为 **Min-Sum 算法**：
1.  **符号相乘**：算出除 $v_j$ 外其他所有节点 LLR 符号的乘积（即简单的异或逻辑）。
2.  **取最小值**：找出除 $v_j$ 外所有节点 LLR 绝对值中**最小的一个**。

将这二者结合发给 $v_j$。这样就把极其复杂的超越函数连乘，变成了极度适合芯片运算的**“异或门 + 比较器”**，使得 LDPC 能够实现 Gbps 级别的超高译码吞吐量。

#### 附录：校验节点更新公式的严格数学推导 (Gallager 引理)

如前所述，BP算法在校验节点的更新本质上是计算多个独立变量模2加（异或）为0时的对数似然比。以下是这个神奇的 $\tanh$ 等式在局部独立假设下的严格推导：

**1. 概率与 LLR 的映射关系**
设二进制随机变量 $X$ 的对数似然比为 $L = \ln \frac{P(X=0)}{P(X=1)}$。
由于 $P(X=0) + P(X=1) = 1$，我们可以解出：
$P(X=0) = \frac{e^L}{1+e^L}, \quad P(X=1) = \frac{1}{1+e^L}$

现在，我们考察两者的**概率差值**：
$P(X=0) - P(X=1) = \frac{e^L - 1}{e^L + 1}$
分子分母同除以 $e^{L/2}$，得到极其关键的一步变换：
$P(X=0) - P(X=1) = \frac{e^{L/2} - e^{-L/2}}{e^{L/2} + e^{-L/2}} = \tanh\left(\frac{L}{2}\right)$
即：任何一个二进制变量“取0的概率减去取1的概率”，正好等于其 LLR 一半的双曲正切。

**2. 两个独立变量的异或 (XOR)**
设有两个独立的二进制变量 $A$ 和 $B$，令 $C = A \oplus B$。根据异或的真值表：
$P(C=0) = P(A=0)P(B=0) + P(A=1)P(B=1)$
$P(C=1) = P(A=0)P(B=1) + P(A=1)P(B=0)$

计算变量 $C$ 的概率差值：
$P(C=0) - P(C=1) = [P(A=0)P(B=0) + P(A=1)P(B=1)] - [P(A=0)P(B=1) + P(A=1)P(B=0)]$
提取公因式，进行因式分解：
$P(C=0) - P(C=1) = (P(A=0) - P(A=1)) \times (P(B=0) - P(B=1))$

**3. 推广到多变量与等式代入**
通过数学归纳法，上述结论可以推广到 $m$ 个独立变量的异或和。设 $Z = X_1 \oplus X_2 \oplus \dots \oplus X_m$：
$P(Z=0) - P(Z=1) = \prod_{i=1}^m (P(X_i=0) - P(X_i=1))$

现在，将第1步得到的 $\tanh$ 映射公式代入上式：
$\tanh\left(\frac{L(Z)}{2}\right) = \prod_{i=1}^m \tanh\left(\frac{L(X_i)}{2}\right)$

两边同时取反函数，即可得到最终异或和变量 $Z$ 的 LLR：
$L(Z) = 2 \tanh^{-1} \left( \prod_{i=1}^m \tanh\left(\frac{L(X_i)}{2}\right) \right)$

**结论与“近似性”探讨**：
将 $Z$ 视作校验节点 $c_i$ 计算出的用于传递给 $v_j$ 的“除 $v_j$ 外其他变量的和”（必须满足它等于 $v_j$ 以确保异或总和为0），我们就得到了步骤4.2中的 SPA 更新公式。**正如你所提及的，这个等式在数学上完全严格的前提是：传入校验节点的所有 $X_i$ 必须是统计独立的。** 在 LDPC 码的 Tanner 图中，只要没有成环（即局部是树状结构），这种独立性就成立。一旦图中存在环，消息会在循环中传递，变量将不再严格独立，此时公式计算出的就不再是全局真实的后验似然比（Exact Likelihood），而是变成了“带环置信传播 (Loopy BP)”的近似似然比。

#### 附录：带环置信传播 (Loopy BP) 的近似度与理论基础

在 Tanner 图存在环（Loop）的情况下，变量不再独立，BP 算法不再计算精确的全局后验概率。然而，Loopy BP 在工程中却表现出惊人的性能。理论界对 Loopy BP 的“近似度”证明主要建立在以下两个里程碑式的理论框架上：

**1. 拓扑视角：计算树展开 (Computation Tree Unrolling)**
*   **理论核心**：R. G. Gallager 和 N. Wiberg 指出，如果在有环图上运行 $l$ 次 BP 迭代，其产生的信息等价于将原图以目标节点为根，向外“展开”成一棵深度为 $l$ 的**计算树 (Computation Tree)**，然后在这棵树上进行精确的（无环）推断。
*   **近似度分析**：在计算树中，原图中的同一个节点可能会在树的不同分支和层级中重复出现多次。Loopy BP 的近似误差来源于它**将这些重复出现的同一节点当作了独立的先验输入**。当图的“围长（Girth，最短环的长度）”很大时，在早期的迭代中，计算树是严格无环的，此时 BP 的结果是**精确等于**真实后验似然的（这也是为什么 LDPC 码设计时要极力最大化围长，通常要求 Girth $\ge 6$ 甚至 $8$）。
*   **渐近精确性 (Density Evolution)**：Richardson 和 Urbanke 证明了，当码长 $N \to \infty$ 时，局部邻域在固定迭代次数 $l$ 下以概率 1 呈树状结构。此时 Loopy BP 的译码阈值（Threshold）是严格解析可证的。

**2. 统计物理视角：Bethe 自由能 (Bethe Free Energy)**
*   **理论核心**：2001年，Yedidia, Freeman 和 Weiss 发表了震撼学术界的论文，证明了 Loopy BP 并非毫无根据的启发式算法，它与统计力学中的能量近似模型有极其深刻的数学等价性。
*   **变分推断**：精确推断本质上是寻找使系统的“吉布斯自由能 (Gibbs Free Energy)”最小化的概率分布。由于有环图的吉布斯自由能无法精确求解，物理学家常使用一种叫 **Bethe 变分近似** 的方法，它只考虑单节点和节点对（边）的局部相互作用，忽略长程相关性。
*   **证明结论**：定理证明，**Loopy BP 算法的信息传递更新公式，等价于在使用拉格朗日乘子法求解 Bethe 自由能的驻点 (Stationary Points)**。并且，**Loopy BP 的任何一个收敛不动点 (Fixed Point)，都严格对应于 Bethe 自由能的一个局部极小值或鞍点**。

**总结**：
Loopy BP 的近似度并没有一个简单的绝对数值界限，而是取决于图的拓扑结构（尤其是短环的数量）和信道的噪声水平。但 Bethe 自由能理论证明了，只要 BP 能够收敛，它找到的就是在局部约束（只看单点和边）下“能量最优”的近似后验分布，这在数学上保证了它作为一种近似算法的合理性与极高的准确度。